# Django, Migrations

## Introduction to Django Migrations
---

In this lesson, you'll learn how **migrations** work in Django.

Migrations are Django's way of propagating changes you make to your models into your database schema.

<br>

🔑 Think of migrations as a version control system (similar to Git) for your database.

While you define what the data should look like in Python within your models (`models.py`), migrations are the way Django "translates" those changes into actual tables in the database (SQL).

#### Why should you care?
Without migrations, every time you added a field to a model, you would have to manually write SQL commands like `ALTER TABLE` ... `ADD COLUMN` ... . Django does this for you automatically and safely.

1. [What are Migrations](#What-are-Migrations),
    - [Why Migrations Matter](#Why-Migrations-Matter),
    - [The Migration Workflow](#The-Migration-Workflow),
2. [Creating Migrations with makemigrations](#Creating-Migrations-with-makemigrations),
    - [Running makemigrations](#Running-makemigrations),
    - [What Gets Created](#What-Gets-Created),
3. [Applying Migrations with migrate](#Applying-Migrations-with-migrate),
    - [Running migrate](#Running-migrate),
    - [Migration Status with showmigrations](#Migration-Status-with-showmigrations),
4. [Working with Migration Files](#Working-with-Migration-Files),
    - [Understanding Migration File Structure](#Understanding-Migration-File-Structure),
    - [Version Control and Migrations](#Version-Control-and-Migrations),
5. [Troubleshooting Migrations](#Troubleshooting-Migrations),
    - [Rolling Back Migrations](#Rolling-Back-Migrations),
    - [Resolving Migration Conflicts](#Resolving-Migration-Conflicts),
    - [Rollback with Existing Data (dumpdata/loaddata)](#Rollback-with-Existing-Data-dumpdataloaddata),
    - [Fake Migrations](#Fake-Migrations),
6. [🧠 EXERCISE 🧠, Create and Apply Migrations](#🧠-EXERCISE-🧠,-Create-and-Apply-Migrations).

<br>

## What are Migrations

---

### Why Migrations Matter

---

Imagine you've been running your blog application for months. The database has thousands of blogs and users.

Now you need to add a new field - `page_count` - to your Book model.

<br>

**Without migrations**, you would have to:
1. Write SQL manually: `ALTER TABLE books ADD COLUMN page_count INTEGER;`
2. Keep track of all changes across development, staging, and production
3. Remember which changes were applied where
4. Hope nothing breaks!

<br>

**With Django migrations**, you just:
1. Change your model in Python
2. Run `makemigrations` - Django detects the change
3. Run `migrate` - Django applies the change
4. The migration file is version controlled - same change everywhere!

<br>

**Migrations provide:**

| Benefit | Description |
|---------|-------------|
| **Version control** | Track database changes like code changes |
| **Reproducibility** | Same migrations = same schema everywhere |
| **Reversibility** | Roll back changes when needed |
| **Team collaboration** | Share database changes through Git |
| **No SQL needed** | Django generates SQL automatically |

<br>

<div style="text-align: center;">
    <img src="../images/arrow.jpeg" width="300" height="200">
</div>


### The Migration Workflow

---

The migration process has **three steps**:

<br>

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│   1. CHANGE     │     │   2. MAKE        │     │   3. APPLY      │
│   models.py     │ ──► │   makemigrations │ ──► │   migrate       │
│                 │     │                  │     │                 │
│   (Python code) │     │   (Creates file) │     │   (Updates DB)  │
└─────────────────┘     └──────────────────┘     └─────────────────┘
```

<br>

1. **Change**: You modify `models.py` (add field, change field, delete model, etc.)
2. **Make**: `makemigrations` detects changes and creates a migration file
3. **Apply**: `migrate` executes the migration and updates the database

<br>

## Creating migration file with `makemigrations`

---

### Running makemigrations

---

The `makemigrations` command scans your models, compares them to the current migration state, and creates new migration files for any changes.

<br>

**Basic usage:**

```bash
# Checks existing migrations in a specific app.
python manage.py showmigrations <app_name>

# Check the query inside the migration file
python manage.py sqlmigrate <app_name> <migration_file>

# Create migrations for all apps
python manage.py makemigrations

# Create migrations for specific app
python manage.py makemigrations catalog

# Create migration with a custom name
python manage.py makemigrations catalog --name add_page_count_to_book
```

<br>

**Example scenario:**

Let's say you have this model:

```python
# blog/models.py (BEFORE)

class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
```

And you add a new field:

```python
# blog/models.py (AFTER)

class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    page_count = models.IntegerField(default=1)  # NEW FIELD
```

<br>

**Running `makemigrations`:**

```bash
$ python manage.py makemigrations blog

Migrations for 'blog':
  blog/migrations/0002_blog_page_count.py
    - Add field page_count to blog
```

Django detected the new field and created a migration file!

<br>

### What Gets Created

---

Migration files are created in the `migrations` directory of your app:

```
blog/
├── __init__.py
├── models.py
├── views.py
└── migrations/
    ├── __init__.py
    ├── 0001_initial.py           # First migration (creates model)
    └── 0002_blog_page_count.py   # Second migration (adds field)
```

<br>

**Migration naming convention:**

| Pattern | Meaning |
|---------|---------|
| `0001_initial.py` | First migration, creates the model |
| `0002_book_page_count.py` | Second migration, auto-named based on change |
| `0003_custom_name.py` | Migration with custom name (`--name` flag) |

<br>

The number prefix ensures migrations are applied in order.

<br>

**Useful makemigrations options:**

```bash
# Preview what would happen (don't create files)
python manage.py makemigrations --dry-run

# See the SQL that would be generated
python manage.py sqlmigrate blog 0003

# Check for model changes without creating migrations
python manage.py makemigrations --check
# Returns exit code 1 if changes detected (useful in CI)
```

<br>

## Applying Migrations with `migrate`

---

### Running migrate

---

The `migrate` command applies pending migrations to your database.

<br>

**Basic usage:**

```bash
# Apply all pending migrations
python manage.py migrate

# Apply migrations for specific app
python manage.py migrate blog

# Apply up to a specific migration
python manage.py migrate blog 0003
```

<br>

**Example output:**

```bash
$ python manage.py migrate

Operations to perform:
  Apply all migrations: admin, auth, catalog, contenttypes, sessions
Running migrations:
  Applying blog.0003_blog_page_count... OK
```

<br>

Django tracks which migrations have been applied in a special table called `django_migrations`.
```bash
18|blog|0001|2026-04-02 05:29:19.326375
29|blog|0002|2026-04-02 13:31:18.048282
30|blog|0003_to_be_removed|2026-04-07 07:47:42.171719
```


<br>

### Migration Status with showmigrations

---

The `showmigrations` command shows the status of all migrations:

```bash
$ python manage.py showmigrations

admin
 [X] 0001_initial
 [X] 0002_logentry_remove_auto_add
 [X] 0003_logentry_add_action_flag_choices
auth
 [X] 0001_initial
 [X] 0002_alter_permission_name_max_length
 ...
catalog
 [X] 0001_initial
 [X] 0002_xyz
 [ ] 0003_blog_page_count              # <-- Not yet applied
```

<br>

| Symbol | Meaning |
|--------|--------|
| `[X]` | Migration has been applied |
| `[ ]` | Migration is pending |

<br>

**Useful options:**

```bash
# Show only for specific app
python manage.py showmigrations catalog

# Show in plan format (order of execution)
python manage.py showmigrations --plan
```

<br>

## Working with Migration Files

---

### Understanding Migration File Structure

---

Let's look at what's inside a migration file:

In [ ]:
# Example: blog/migrations/0001_initial.py

from django.db import migrations, models


class Migration(migrations.Migration):
    
    initial = True

    # Dependencies - which migrations must be applied first
    dependencies = [
    ]
    
    # Operations - what this migration does
    operations = [
        migrations.CreateModel(
            name='Blog',
            fields=[
                ('id', models.BigAutoField(auto_created=True, primary_key=True, serialize=False, verbose_name='ID')),
                ('title', models.CharField(max_length=200)),
                ('author', models.CharField(max_length=100)),
                ('published_date', models.DateField()),
            ],
        ),
    ]

<br>

**Mandatory key components:**

| Component | Description |
|-----------|-------------|
| `dependencies` | List of migrations that must run first |
| `operations` | List of changes to apply |

<br>

**Common operations:**

| Operation | What it does |
|-----------|-------------|
| `CreateModel` | Create a new table |
| `DeleteModel` | Delete a table |
| `AddField` | Add a column |
| `RemoveField` | Remove a column |
| `AlterField` | Change a column |
| `RenameField` | Rename a column |
| `AddIndex` | Create an index |
| `RemoveIndex` | Remove an index |

<br>

### Version Control and Migrations

---

**Always commit migration files to Git!**

Migrations are part of your codebase - they ensure every environment (development, staging, production) has the same database schema.

<br>

```bash
# After creating migrations
git add blog/migrations/
git commit -m "Add page_count field to Blog model"
```

<br>

**What to commit:**

| Include | Exclude |
|---------|--------|
| Migration files (`*.py`) | `__pycache__` |
| `__init__.py` in migrations | Database files (`.sqlite3`) |

<br>

**Typical workflow with Git:**

```bash
# 1. Pull latest code (includes migrations from teammates)
git pull origin main

# 2. Apply any new migrations
python manage.py migrate

# 3. Make your model changes
# ... edit models.py ...

# 4. Create migration for your changes
python manage.py makemigrations

# 5. Test locally
python manage.py migrate
python manage.py runserver

# 6. Commit everything
git add .
git commit -m "Add new feature with model changes"
git push
```

<br>

## Troubleshooting Migrations

---

### Rolling Back Migrations

---

Made a mistake? You can **roll back** to a previous migration:

```bash
# Roll back to a specific migration
python manage.py migrate blog 0001

# This will REVERSE all migrations after 0001
```

<br>

**Example:**

```bash
$ python manage.py showmigrations blog
blog
 [X] 0001_initial
 [X] 0002_blog_page_count      # We want to undo this
 [X] 0003_blog_publisher

$ python manage.py migrate blog 0001
Operations to perform:
  Target specific migration: 0001_initial, from blog
Running migrations:
  Rendering model states... DONE
  Unapplying blog.0003_blog_publisher... OK
  Unapplying blog.0002_blog_page_count... OK

$ python manage.py showmigrations blog
blog
 [X] 0001_initial
 [ ] 0002_book_page_count      # Now unapplied
 [ ] 0003_book_publisher
```

<br>

**To completely unapply all migrations for an app:**

```bash
python manage.py migrate catalog zero
```

⚠️ **Warning:** This removes all tables for that app!

<br>

### Resolving Migration Conflicts

---

We would like to add a new field to the `Blog` model.

But our colleague is also working on the same model and created a migration for a different field.

In [ ]:
from django.db import models

class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    page_count = models.IntegerField(default=1)  # ours
    status = models.CharField(max_length=10,     # their
                              choices=('draft', 'published', 'archived'),
                              default='draft')

**Scenario:** Two developers create migrations at the same time in app `blog`:

```
Developer A creates: 0003_blog_count.py
Developer B creates: 0003_blog_status.py

Both depend on 0002.py.
```

...and now with following command:

```bash
python manage.py migrate blog
CommandError: Conflicting migrations detected; multiple leaf nodes in the migration graph: (0003_blog_count, 0003_blog_status in blog).
To fix them run 'python manage.py makemigrations --merge'
```

<br>

**Live demo commands (simulate this exact conflict):**

```bash
# Check pending migrations in blog app
python manage.py showmigrations blog

# Confirm conflict by trying to migrate
python manage.py migrate blog

# Optional: inspect both conflicting files
python manage.py sqlmigrate blog 0003_blog_count
python manage.py sqlmigrate blog 0003_blog_status
```

**Hint:** Conflict appears when two migrations have the same parent (`('blog', '0002')`) and both are leaf nodes.

<br>

**Solution 1: Merge migrations**

```bash
# 1. Create merge migration for app blog
python manage.py makemigrations blog --merge

# 2. Inspect merge result
python manage.py showmigrations blog --plan

# 3. Apply
python manage.py migrate blog

# 4. Commit migration files
git add mysite/blog/migrations/
git commit -m "Merge blog migration branches (0003_blog_count + 0003_blog_status)"
```

This creates a merge migration similar to:

```python
# mysite/blog/migrations/0004_merge_*.py

class Migration(migrations.Migration):
    dependencies = [
        ('blog', '0003_blog_count'),
        ('blog', '0003_blog_status'),
    ]
    operations = []
```

**Hint for students:** `operations = []` is normal in merge migration; it only reconnects the migration graph.

**Pitfall to mention on lesson:** always verify dependencies before running merge on shared or production environments.

<br>

**Solution 2: Recreate one migration (only before push)**

Use this when one of the two conflicting migrations is only local and not shared yet.

```bash
# 1. Remove local conflicting migration (example: count)
rm mysite/blog/migrations/0003_blog_count.py

# 2. Keep teammate migration as source of truth
git pull

# 3. Apply teammate migration
python manage.py migrate blog

# 4. Recreate your change as a new migration (likely 0004_*)
python manage.py makemigrations blog
python manage.py migrate blog
```

**Hint** Never rewrite already shared migrations in team branches. If migration was pushed, prefer merge migration.

**Solution 3:** You can even change the filename and dependency variable manually.

<br>

### Rollback with Existing Data (dumpdata/loaddata)

---

When rollback can affect existing rows, make a backup first and restore after schema is stabilized.

<br>

**Live demo commands (safe rollback with data backup):**

```bash
# 0) Optional: verify current state
python manage.py showmigrations blog
python manage.py shell -c "from blog.models import Blog; print(Blog.objects.count())"

# 1) Backup data from app before rollback
mkdir -p backups
python manage.py dumpdata --indent 2 > backups/blog_before_rollback.json

# 2) Roll back one step (example: from 0003 to 0002)
python manage.py migrate blog 0002

# 3) Update models/migrations if needed
# ... adjust model or create corrective migration ...
python manage.py makemigrations blog
python manage.py migrate blog

# 4) Reload backup data (if data was removed/changed)
python manage.py loaddata backups/blog_before_rollback.json

# 5) Verify
python manage.py shell -c "from blog.models import Blog; print(Blog.objects.count())"
python manage.py showmigrations blog
```

**Best practices for production:**
- Prefer DB-native backup + `dumpdata` for critical tables.
- Use `--natural-foreign --natural-primary` when fixtures contain many relations.
- Avoid `migrate <app> zero` on shared/production DB unless fully planned.

**Common pitfall:** `loaddata` can fail on unique constraints if rows already exist. In that case, restore into a clean target table or use a controlled data migration script.

<br>

### Fake Migrations

---

Sometimes the database already has the changes, but Django doesn't know it. Use `--fake` to mark a migration as applied without running it:

```bash
# Mark migration as applied without executing it
python manage.py migrate catalog 0002 --fake
```

<br>

**Common use cases:**

| Scenario | Solution |
|----------|----------|
| Database was modified manually | `--fake` the migration |
| Starting fresh with existing schema | `--fake-initial` |
| Fixing migration state after manual fix | `--fake` specific migration |

<br>

Create a new table manually:
```sql
# creat via dbshell
CREATE TABLE blog_author (
    id integer NOT NULL PRIMARY KEY AUTOINCREMENT,
    name varchar(100) NOT NULL,
    bio text NOT NULL
);
```

Add a new model:
```python
# mysite/blog/models.py
from django.db import models

class Author(models.Model):
    name = models.CharField(max_length=100)
    bio = models.TextField()
```

Create the migration file:
```bash
python manage.py makemigrations blog
```

...and finally try to apply migration file:
```bash
python manage.py migrate blog
# django.db.utils.OperationalError: table "blog_author" already exists
```

Now we have 2 options:
1. either delete the EMPTY table in the database,
2. or tell Django, its done

**Check the filename**
```bash
python manage.py showmigrations blog
# ...
# [X] 0004_blog_count
# [ ] 0005_author
```

Create a fake migration file:
```bash
uv run python manage.py migrate blog 0005 --fake 
Operations to perform:
  Target specific migration: 0005_author, from blog
Running migrations:
  Applying blog.0005_author... FAKED
```

🔑 Final result.

<br>

**Example: New project with existing database**

```bash
# Tables already exist in database
# Mark initial migrations as applied without creating tables
python manage.py migrate --fake-initial
```

<br>

## Common Migration Scenarios

---

### Adding a Required Field to Existing Data

---

When you add a required field to a model that already has data, Django will ask what to do:

```bash
$ python manage.py makemigrations

You are trying to add a non-nullable field 'isbn' to book without a default.
We can't do that (the database needs something to populate existing rows).
Please select a fix:
 1) Provide a one-off default now (will be set on all existing rows)
 2) Quit, and let me add a default in models.py
Select an option:
```

<br>

**Options:**

1. **Option 1:** Enter a value now (e.g., `'unknown'` for a CharField)
2. **Option 2:** Add `default=...` or `null=True` in `models.py` first

<br>

🔑 **Best practice (three-step migration):** Add field as nullable, populate via data migration, then make it required.

---

**Example:** Add `status` field with choices to existing `Blog` model.

**Step 1: Add field as nullable**

```python
# blog/models.py
class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    status = models.CharField(max_length=10,
                              choices=(('draft', 'Draft'), ('published', 'Published'), ('archived', 'Archived')),
                              null=True, blank=True)  # Temporary nullable

```

```bash
python manage.py makemigrations blog --name add_status_nullable
python manage.py migrate blog
```

**Step 2: Populate existing rows with data migration**

Create empty migration:

```bash
python manage.py makemigrations blog --empty --name populate_blog_status
```

Edit the migration file to add logic:

```python
# blog/migrations/0005_populate_blog_status.py

from django.db import migrations

def populate_status(apps, schema_editor):
    Blog = apps.get_model('blog', 'Blog')
    
    # Define status mapping by ID
    status_map = {
        1: 'published',  # Django Bootcamp
        2: 'draft',      # My Blog
        3: 'draft',      # My Blog
        4: 'published',  # My Super Blog
        5: 'published',  # Second attempt
        6: 'published',  # Fast API Introduction
        7: 'archived',   # Data structures
    }
    
    # Apply mapping
    for blog in Blog.objects.all():
        blog.status = status_map.get(blog.id, 'draft')
        blog.save()
    
    # Alternative: bulk update by ID range
    # Blog.objects.filter(id__in=[1, 4, 5, 6]).update(status='published')
    # Blog.objects.filter(id__in=[2, 3]).update(status='draft')
    # Blog.objects.filter(id=7).update(status='archived')

def reverse_populate(apps, schema_editor):
    # Optional: clear status on rollback
    Blog = apps.get_model('blog', 'Blog')
    Blog.objects.update(status=None)

class Migration(migrations.Migration):
    dependencies = [
        ('blog', '0003_add_status_nullable'),
    ]
    
    operations = [
        migrations.RunPython(populate_status, reverse_populate),
    ]
```

Apply:

```bash
python manage.py migrate blog
```

**Step 3: Make field required**

```python
# blog/models.py
status = models.CharField(
    max_length=10,
    choices=Status.choices,
    default=Status.DRAFT,  # Now with default
    db_index=True,
)
# Remove null=True, blank=True
```

```bash
python manage.py makemigrations blog --name make_status_required
python manage.py migrate blog
```

**Key points for students:**
- `apps.get_model()` gives historical model state (safe for migrations)
- `RunPython` accepts forward + reverse functions
- Never import models directly in migrations (use `apps.get_model`)
- Data migration runs between schema changes
- Alternative logic: populate by ID range, random assignment, external API, etc.

<br>

### Renaming a Field

---

Django is smart about detecting field renames:

```python
# Before
class Blog(models.Model):
    name = models.CharField(max_length=200)

# After
class Blog(models.Model):
    title = models.CharField(max_length=200)  # Renamed
```

```bash
$ python manage.py makemigrations

Did you rename blog.name to blog.title (a CharField)? [y/N] y

Migrations for 'catalog':
  catalog/migrations/0004_rename_name_blog_title.py
    - Rename field name on book to title
```

Django will use `RenameField` operation, preserving your data!

<br>

## Summary

---

| Command | Description |
|---------|-------------|
| `makemigrations` | Create migration files from model changes |
| `migrate` | Apply migrations to database |
| `showmigrations` | Show migration status |
| `sqlmigrate` | Show SQL for a migration |
| `migrate <app> <migration>` | Migrate to specific version |
| `migrate --fake` | Mark as applied without running |
| `makemigrations --merge` | Merge conflicting migrations |

<br>

**Key takeaways:**

- Always run `makemigrations` after changing models
- Always commit migration files to version control
- Run `migrate` on every environment (dev, staging, production)
- Use `showmigrations` to check status
- Handle conflicts with `--merge` or by recreating

<br>

### 🧠 EXERCISE 🧠, Create and Apply Migrations

---

Practice the migration workflow:

1. Create a new model `Author` in your blog app with:
   - `name` (CharField, max 100)
   - `email` (EmailField, optional)
   - `bio` (TextField, optional)

2. Create a migration for the new model

3. View the SQL that would be executed (don't apply yet)

4. Apply the migration

5. Add an `author_profile` ForeignKey field to the `Blog` model (optional, nullable)

6. Create and apply the migration for this change

7. Check migration status with `showmigrations`

<details>
    <summary>▶️ Solution</summary>
    
**1. Edit `blog/models.py`:**

```python
class Author(models.Model):
    name = models.CharField(max_length=100)
    email = models.EmailField(blank=True)
    bio = models.TextField(blank=True)

    def __str__(self):
        return self.name
```

**2. Create migration:**

```bash
python manage.py makemigrations blog
# Output: blog/migrations/0007_author.py
```

**3. View SQL:**

```bash
python manage.py sqlmigrate blog 0007
```

**4. Apply migration:**

```bash
python manage.py migrate blog
```

**5. Add to Blog model:**

```python
class Blog(models.Model):
    # ... existing fields ...
    author_profile = models.ForeignKey(
        Author,
        on_delete=models.SET_NULL,
        null=True,
        blank=True,
        related_name='blogs'
    )
```

**6. Create and apply:**

```bash
python manage.py makemigrations blog
python manage.py migrate blog
```

**7. Check status:**

```bash
python manage.py showmigrations blog
# blog
#  [X] 0001_initial
#  ...
#  [X] 0007_author
#  [X] 0008_blog_author_profile
```
</details>

<br>

### 🧠 EXERCISE 🧠, Handle Migration Rollback

---

Practice rolling back migrations:

1. Check current migration status with `showmigrations`

2. Roll back the last migration you applied

3. Verify the rollback with `showmigrations`

4. Re-apply the migration

5. (Optional) Try rolling back to `zero` and then migrating back up

<details>
    <summary>▶️ Solution</summary>
    
```bash
# 1. Check current status
python manage.py showmigrations catalog
# catalog
#  [X] 0001_initial
#  [X] 0002_publisher
#  [X] 0003_book_publisher

# 2. Roll back to 0002 (unapplies 0003)
python manage.py migrate catalog 0002

# 3. Verify
python manage.py showmigrations catalog
# catalog
#  [X] 0001_initial
#  [X] 0002_publisher
#  [ ] 0003_book_publisher    <-- Now unapplied

# 4. Re-apply
python manage.py migrate catalog

# 5. (Optional) Roll back everything and re-apply
python manage.py migrate catalog zero
python manage.py showmigrations catalog  # All should be [ ]
python manage.py migrate catalog
python manage.py showmigrations catalog  # All should be [X]
```
</details>

---